# Welcome to your new Jupyter notebook!
You can use Python to study the results of the analyses performed on the trajectories you have uploaded to the platform, using libraries such as "pandas", "numpy", among others.

This is achieved using Voltsdk, which provides VoltClient for Python. Through VoltClient, you can interact with your Volt instance's API using a SECRET_KEY that you must create, copy, and paste.

Using the SECRET_KEY, you can access all your team's resources from Python using VoltClient, including your simulations, plugin results, exports, and more.


`NOTE`: This is not a Python tutorial, and I assume that if you are reading this, you already know at least the basics to interact with the Volt API to analyze your results. Otherwise, you can use the UI to download your analysis results individually and analyze them later as you normally would.


## Getting Started
Before you begin, you need to fill in the global variables you will use in the notebook with the correct information.

In [ ]:
# This corresponds to the endpoint of your Volt instance.
# Example: https://server.voltcloud.dev/api
#          https://127.0.0.1:8000/api
BASE_URL = "<BASE_URL>/api"

# The ID of the simulation we will use to analyze
# the results after running the analysis.
TRAJECTORY_ID = "<TRAJECTORY_ID>"

# With the SECRET_KEY you can freely access
# your team's resources. Do not share this key!
SECRET_KEY = "vsk_replace_with_your_secret_key"

## Creating a client instance using voltsdk
To connect to your Volt instance, you must use VoltClient (from voltsdk) and provide the constructor with BASE_URL (your instance's endpoint) and your SECRET_KEY.

In [ ]:
from voltsdk import VoltClient

client = VoltClient(
    secret_key=SECRET_KEY,
    base_url=BASE_URL
)

## List all trajectory analyses
After creating our VoltClient instance, we can use the available methods. Next, using the "list_analyses" method, we can obtain all available analyses by providing the trajectory ID (defined at the beginning).

In [ ]:
analyses = client.list_analyses(trajectory_id=TRAJECTORY_ID)

The variable name indicates that the result is an array. Specifically, an array of dictionaries, where each value corresponds to a specific analysis. Consequently, we can iterate through it.

In [ ]:
for analysis in analyses:
    print('-' * 30)

    print(f'  Analysis ID: {analysis["_id"]}')
    print(f'  Plugin ID: {analysis["plugin"]}')
    print(f'  Plugin Display Name: {analysis["pluginDisplayName"]}')
    print(f'  Created At: {analysis["createdAt"]}')

    # analysis["config"] is a dictionary that corresponds to the
    # configuration provided by the user to run the binary.
    #
    # Each key corresponds to an argument, and the value is the
    # one assigned when the plugin is executed.
    #
    # For example:
    # analysis["config"] = { identificationMode: 'PTM', rmsd: 0.10 }
    #
    # This means that when the user executed the modifier, they did so with
    # the configuration visible in the example, and the call was
    # executed as follows:
    # ./somePlugin --identificationMode PTM --rmsd 0.10
    print('  Config:')
    for argument, value in analysis.get('config').items():
        print(f'    - {argument}: {value}')

    # An analysis can have one or more "exposures". In Volt, an exposure
    # corresponds to a file containing data after an analysis has been run.
    #
    # For example:
    # The exposures in "Dislocation Analysis" include: Interface Mesh,
    # Defect Mesh, Structure Stats, and Dislocations. Each of these represents
    # the resulting information after the analysis is executed, which can
    # be accessed and displayed on screen (as in Volt) and analyzed further.
    print(' Exposures:')
    results = client.list_analysis_results(analysis_id=analysis.get('_id'))
    for result in results:
        print('    - Exposure ID:', result.get('exposureId'))
        print('    - Exposure Name:', result.get('exposureName'))

        # The "Listing Rows" correspond to the information that the plugin
        # has configured to list in tables within the application using
        # the information from the exported files.
        print('    - Listing Rows:')
        for column_name, column_value in result.get('row').items():
            print(f'      - {column_name}: {column_value}')

    print('-' * 30)

## Downloading all analysis information locally
This is useful because in the previous code cell, I mentioned that "Listing Rows" corresponds to the configuration provided by the plugin to display the information for each exposure in tables within Volt.

However, not all the information resulting from the analysis is displayed in the table. This isn't necessarily a problem that needs fixing; you don't want to see low-relevance information in the application's tables.

By downloading the information from all the executed analyses, you can transform the data to study it using libraries like pandas, NumPy, etc.

In addition to downloading the results generated by the analyses, all the 3D models you see within the application are also downloaded, in this case, to view them in this notebook.

In [ ]:
for analysis in analyses:
    client.download_analysis_artifacts(
        analysis_id=analysis.get('_id')
    )

The "download_analysis_artifacts" method of VoltClient will download all the information related to the specified analysis into the "download/" directory in the current working directory.

The structure of the "downloads/" directory will be as follows:

```
downloads/
  analysis-{analysisId}-plugin-{pluginName}/
    plugins/
      trajectory-{trajectoryId}/
        {exposure-id}/
          timestep-{timestep}.msgpack
    trajectory-{trajectoryId}/
      {exposureId}.glb

```

## Using pandas
Volt's philosophy is to be flexible and modular, allowing you to implement any type of algorithm for use in your simulations. However, to achieve this, we must establish certain rules that you, as a developer, must follow when integrating your software with the platform.

One of these rules is that you must export your algorithm's results in "msgpack" format, which is simply a compressed JSON file.

Using voltsdk, we can take these exported msgpack files (see the previous point) and load them with pandas.


In [ ]:
from voltsdk import msgpack_as_df

MSGPACK_FILE_PATH = '...'

df = msgpack_as_df(
    file_path=MSGPACK_FILE_PATH,
    iterable_key='data'
)

The `msgpack_as_df` function takes two parameters: `file_path` and `iterable_key`. The first, as you might guess, is the file path; the second is more interesting. We've already mentioned that the `msgpack` format is compressed JSON, and you should also know that JSON is the equivalent of an object in JavaScript or a `dict` in Python. Consequently, in both cases, we have the same concept: `keys`. Within the resulting JSON, we have several keys after the analysis is performed.

For example, in the "Dislocation Analysis" plugin, among several exposures, we have one called "Dislocations." If you've reached this point, you should already know what an "exposure" is in Volt. Therefore, knowing that we're dealing with an "msgpack" file, we ask ourselves the following question: How can we determine its internal structure so we can use it within the UI? That's why each exposure has a "schema," which, as its name suggests, allows us to describe the structure of the exposure file. Below is the schema for the "Dislocations" exposure:


```json
{
  "metadata": {
    "count": "int"
  },
  "summary": {
    "total_points": "int",
    "average_segment_length": "float",
    "max_segment_length": "float",
    "min_segment_length": "float",
    "total_length": "float"
  },
  "data": {
    "type": "array",
    "items": {
      "segment_id": "int",
      "length": "float",
      "num_points": "int",
      "burgers": {
        "vector": {
          "type": "array",
          "items": "float"
        },
        "magnitude": "float",
        "fractional": "string"
      }
    }
  }
}
```

If you look closely, you'll see that the file contains "metadata", "summary", and "data". The question is: Which of these keys contains the exposure results? In this case, the correct answer is "data". The keys "metadata" and "summary" contain statistics for the dataset, which is located within "data". Therefore, that's the key you should specify to the "msgpack_as_df" function.

The value of the "iterable_key" parameter will NOT always be "data", as this isn't within Volt's rules for developers to ensure compatibility between their software and ours. However, in most cases it will be, since we define it as a "best practice"/"standard".

`NOTE`: The fact that "data" isn't required for the exposure results isn't something that needs fixing in the future. It's not a bug or a design flaw. The plugin developer, by defining its schema, tells Volt how to access the data and display it in the UI. However, you don't have that abstraction layer here at the client level when interacting with the API. That's why you need to manually provide that key!



## Visualizing 3D models
You can view the same 3D models you see in the application, here on your notebook.


In [ ]:
from voltsdk import view_glb

GLB_FILE_PATH = '...'

view_glb(file_path=GLB_FILE_PATH)